# 🎓 Đồ Án Tốt Nghiệp: Huấn Luyện RLVR (GRPO + LoRA) Trên DeepSeek-R1-1.5B

Notebook này phục vụ thực nghiệm **Nhánh B (Reinforcement Learning with Verifiable Rewards - RLVR)**:
- **Mô hình**: `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`
- **Thuật toán**: **GRPO** (*Group Relative Policy Optimization*) kết hợp **LoRA** (PEFT)
- **Tối ưu phần cứng**: Hỗ trợ QLoRA 4-bit chạy mượt mà trên **Google Colab GPU T4 miễn phí** (15GB VRAM) hoặc GPU A100/L4.
- **Quy trình chuẩn**: Trích xuất dữ liệu (Zero Data Leakage) -> Huấn luyện GRPO -> Đánh giá đối chứng 100 câu GSM8K -> Thực nghiệm tổng hợp 4 kịch bản.

## 1. Kiểm tra thông tin GPU và Cài đặt Thư viện

In [ ]:
!nvidia-smi

In [ ]:
# Cài đặt các thư viện mới nhất cần thiết cho GRPO, LoRA và lượng tử hóa 4-bit
!pip install -q trl>=0.12.0 peft>=0.7.0 bitsandbytes>=0.41.0 datasets>=2.16.0 accelerate>=0.26.0 transformers>=4.48.0

## 2. Chuẩn bị Dữ liệu Huấn luyện (GSM8K Train Split - Zero Data Leakage)

Trích xuất 400 câu từ tập train gốc của GSM8K, tự động đối chiếu và loại bỏ bất kỳ câu nào trùng với 100 câu kiểm thử trong `common/data/gsm8k_test_100.json`.

In [ ]:
!python method_b_rlvr/prepare_train_data.py --num_samples 400 --output_file method_b_rlvr/data/gsm8k_train_split.json

## 3. Khởi chạy Huấn luyện GRPO + LoRA

- Trên **GPU T4**: Bật cờ `--use_4bit` để tiết kiệm VRAM tối đa (~1.2GB VRAM mô hình nền).
- Trên **GPU A100 / L4**: Có thể bỏ cờ `--use_4bit` để chạy trực tiếp trên độ chính xác `bfloat16`.

In [ ]:
!python method_b_rlvr/train_grpo.py \
    --model_id deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --data_file method_b_rlvr/data/gsm8k_train_split.json \
    --num_samples 400 \
    --num_generations 4 \
    --learning_rate 5e-6 \
    --batch_size 1 \
    --gradient_accumulation_steps 4 \
    --max_prompt_length 256 \
    --max_completion_length 512 \
    --epochs 1 \
    --lora_r 16 \
    --lora_alpha 32 \
    --use_4bit \
    --output_dir results/grpo_deepseek_r1_lora \
    --adapter_dir results/rlvr_adapter

## 4. Đánh giá Mô hình sau RLVR trên 100 câu GSM8K Kiểm thử

So sánh trực tiếp kết quả với Baseline gốc (59.0%) và RPDI-EE (74.0%).

In [ ]:
!python method_b_rlvr/evaluate.py \
    --model_id deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --adapter_dir results/rlvr_adapter \
    --output_file results/method_b_results.json

## 5. Thực nghiệm Tổng hợp 4 Kịch bản Đối chứng (Luận văn Tốt nghiệp)

Chạy đồng thời trên cùng 100 câu GSM8K để xuất bảng so sánh 4 kịch bản:
1. **KB1**: Baseline Gốc (Không can thiệp)
2. **KB2**: Baseline + RPDI-EE (Phanh ngoại vi)
3. **KB3**: RLVR (Học tăng cường nội vi qua GRPO)
4. **KB4**: RLVR + RPDI-EE (Kết hợp cả 2 phương pháp)

In [ ]:
!python combined/run_4_scenarios.py \
    --model_id deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --adapter_dir results/rlvr_adapter \
    --W 8 \
    --lambda_th 1.2 \
    --min_steps 64 \
    --max_new_tokens 768 \
    --output_file results/combined_4_scenarios.json